# Data Extraction

In [ ]:
# Run if dependencies not installed
%pip install numpy
%pip install scikit-learn
%pip install matplotlib
%pip install pandas
%pip install seaborn

### Extracting the Training and Test Data

In [ ]:
import pandas as pd

train_df = pd.read_csv("../data/splits/train.csv")
test_df  = pd.read_csv("../data/splits/test_github.csv")

print("=" * 30)
print("TRAIN SET")
print("=" * 30)
print(f"Shape : {train_df.shape}\n")
print(f"Column names:\n{train_df.columns.tolist()}\n")
print(f"Data types:\n{train_df.dtypes}\n")
print(f"First 5 rows:\n{train_df.head()}\n")

print("=" * 30)
print("TARGET")
print("=" * 30)
print(train_df["readmitted"].value_counts())
print(train_df["readmitted"].value_counts(normalize=True))

### Removing Dupicates from Data Set
There are no duplicates in the training data

In [ ]:
import pandas as pd

print("=" *60)
num_duplicates  = train_df.duplicated().sum()
print(f"The train set contains: {num_duplicates } duplicates")
print("=" *60)

if (num_duplicates > 0):
    train_df = train_df.drop_duplicates()
    print(train_df.describe())

### Determining the Distribution of Training Data

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

sns.histplot(train_df.age, kde=True).set(title="Ages Distribution")
plt.xticks(rotation=45, ha='right')
plt.show()
sns.histplot(train_df.gender).set(title="Gender Distribution")
plt.show()
sns.histplot(train_df.race).set(title="Race Distribution")
plt.xticks(rotation=45, ha='right')
plt.show()
sns.histplot(train_df.weight).set(title="Weight Distribution")
plt.xticks(rotation=45, ha='right')
plt.show()
sns.histplot(train_df.number_diagnoses).set(title="Other Diagnoses Distribution")
plt.xticks(rotation=45, ha='right')
plt.show()


In [ ]:
plt.figure(figsize=(8, 5))
train_df["readmitted"] = pd.Categorical(train_df["readmitted"], ["<30", ">30", "No"])

ax = sns.countplot(
    x="readmitted",
    data=train_df,
    order=["<30", ">30", "No"],
)

for p in ax.patches:
    ax.annotate(f'{int(p.get_height())}',
                (p.get_x() + p.get_width() / 2, p.get_height()),
                ha='center', va='bottom')

plt.title("Readmission Distribution (Training Set)")
plt.xlabel("Readmission Status")
plt.ylabel("Number of Patients")

plt.tight_layout()
plt.show()

In [ ]:
row_count = train_df.shape[0]
for feature in train_df:    
    count = (train_df[feature] == "?").sum()
    count += (train_df[feature].isnull()).sum()
    print(f"Missing value count [{feature}]: {count}, percentage: {round(count/row_count, 3)}%")

### Summary of Distributions

### Visulalizing Correlations

In [ ]:
import pandas as pd
import seaborn as sns

#testign
numerics = train_df.select_dtypes(include="number")
correlation = train_df.corr(numeric_only=True)

plt.figure()

sns.heatmap(correlation)

plt.show()


This correlation plot does not tell us a lot and the reason for this is that a lot of the data in the training set is categorical. To extract some meaningful information we have to do some preprocessing of the data.

# Preprocessing pipeline

### Dropping uneccesarry columns

In [ ]:
# Essentially these columns are just identifiers.
COLUMNS_TO_DROP = ["id", "encounter_id", "patient_nbr"]
train_df.drop(columns=COLUMNS_TO_DROP, inplace=True)

In [ ]:
# If a column contains more than THREASHOLD missing data then we will remove that column
THREASHOLD = 0.10 # In as a percentage so 0.10 is 10%
# The selected threshold of 10% is selected on the basis of this paper:
# https://www.sciencedirect.com/science/article/pii/S1326020023036488
# Which argues that a feature with more than 10% missing values can lead to bias


row_count = train_df.shape[0]
columns_with_missing_data = []

for feature in train_df:
    if feature == "readmitted":
        continue
    count = (train_df[feature] == "?").sum()
    count += (train_df[feature].isnull()).sum()
    percentage_missing = count/row_count
    if percentage_missing > THREASHOLD:
        columns_with_missing_data.append(feature)

print(columns_with_missing_data)
train_df.drop(columns=columns_with_missing_data, inplace=True)


### Reduce target variable

In [10]:
# Reduce 3-class target to binary class
# The most clinically intressting is <30 so that is all patients readmitted within 30 days
# But with the assignment saying "patient that gets readmitted or not this would be more inline with the
# project description.
target = "readmitted"
train_df[target] = train_df[target].map(lambda x: 1 if x == '<30' or x == '>30' else 0)
print(train_df[target].value_counts())
print(train_df[target].unique())
print(train_df[target].sum()) 

### Replace remaining missing values '?' with NaN

In [ ]:
train_df.replace('?', pd.NA, inplace=True)

### One-Hot-Encoding

In [ ]:
from sklearn import preprocessing
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

enc = preprocessing.OneHotEncoder(handle_unknown="ignore", sparse_output=False)
enc.set_output(transform="pandas")
cat_cols = train_df.select_dtypes(include=["object", "category"]).columns.to_list()

encoded = enc.fit_transform(train_df[cat_cols])

full_df = pd.concat([encoded, train_df.select_dtypes(include="number")], axis=1)

cols_of_interest = [c for c in full_df.columns if c.startswith("race_")]
cols_of_interest.append("number_diagnoses")
cols_of_interest.append("readmitted")

corr = full_df[cols_of_interest].corr()

plt.figure()
sns.heatmap(corr, cmap=sns.cubehelix_palette(as_cmap=True))
plt.show()

### Remove columns with zero variance

In [ ]:
zero_variance_cols = [
    col for col in train_df.columns
    if col != 'readmitted' and train_df[col].nunique() <= 1
]

print(f"Zero-variance columns found: {zero_variance_cols}")
train_df.drop(columns=zero_variance_cols, inplace=True)

### ICD-9 numerical values to Categorical groups

In [ ]:
def map_icd_to_category(code):
    if code is None:
        return 'Other'
    try:
        c = float(code)
    except:
        return 'Other'
    if 390 <= c <= 459 or c == 785:   return 'Circulatory'
    if 460 <= c <= 519 or c == 786:   return 'Respiratory'
    if 520 <= c <= 579 or c == 787:   return 'Digestive'
    if c == 250:                       return 'Diabetes'
    if 800 <= c <= 999:                return 'Injury'
    if 710 <= c <= 739:                return 'Musculoskeletal'
    if 580 <= c <= 629 or c == 788:   return 'Genitourinary'
    if 140 <= c <= 239:                return 'Neoplasms'
    return 'Other'

for col in ['diag_1', 'diag_2', 'diag_3']:
    train_df[col] = train_df[col].apply(map_icd_to_category)

train_df[['diag_1', 'diag_2', 'diag_3']].head()